In [ ]:
!pip install transformers torch

In [ ]:
!pip install datasets tokenizers

In [ ]:
!pip install streamlit

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer,pipeline
from datasets import Dataset, load_dataset
import torch

In [ ]:
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json

In [ ]:
import json
with open("train-v1.1.json", "r") as f:
    sdata = json.load(f)


In [ ]:
data = []

for article in sdata["data"]:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            question = qa["question"]
            answer = qa["answers"][0]["text"]
            data.append({"context": context, "question": question, "answer": answer})

df = pd.DataFrame(data)
df = df.sample(100,random_state=42)
df.head()

In [ ]:
	qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    tokenizer="distilbert-base-cased",
    device=-1
)
sample = df.iloc[0]
context = sample["context"]
question = sample["question"]
result = qa_pipeline(context=context, question=question)

print(f"Question: {question}")
print(f"Predicted Answer: {result['answer']}")
print(f"True Answer: {sample['answer']}")

In [ ]:
preds = []
for i in range (10):
    row = df.iloc[i]
    result = qa_pipeline(context=row["context"], question=row["question"])
    preds.append({
        'questions' : row['question'],
        'predicted_answer' : result['answer'],
        'true_answer' : row['answer']
    })
pd.DataFrame(preds)

In [ ]:
import string
def normalize_text(text):
    text = text.lower()
    text = "".join(char for char in text if char not in string.punctuation)
    return text

def CAL_exact_match(pred,true):
    return int(normalize_text(pred) == normalize_text(true))

def CAL_F1_score(pred,true):
    pred_tokens = normalize_text(pred).split()
    true_tokens = normalize_text(true).split()

    common = set(pred_tokens) & set(true_tokens)
    if not common:
        return 0.0

    pre = len(common) / len(pred_tokens)
    rec = len(common) / len(true_tokens)

    F1_score = 2 * (pre * rec) / (pre + rec)
    return F1_score

In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    tokenizer="distilbert-base-cased")
exact_match=[]
F1_score = []

for i in range (min(10,len(df))):
    row = df.iloc[i]
    result = qa_pipeline(context=row["context"], question=row["question"])
    pred = result['answer']
    true = row['answer']

    exact_match.append(CAL_exact_match(pred,true))
    F1_score.append(CAL_F1_score(pred,true))

print("Exact Match :",sum(exact_match)/len(exact_match))
print("F1 Score :",sum(F1_score)/len(F1_score))

In [ ]:
def evaluate_model(model_name,df,n=50):
  print("Evaluate model",{model_name})
  qa_pipeline = pipeline("Question-Answering",model = model_name)

  exact_match,F1 = [], []
  for i in range(n):
    row = df.iloc[i]
    result = qa_pipeline(context = row['context'],question = row['question'])
    pred = result['answer']
    true = row['answer']

    exact_match.append(exact_match(pred,true))
    F1.append(F1(pred,true))

  em = np.mean(exact_match)
  f1 = np.mean(F1)

  print("Exact Match :",em)
  print("F1 Score :",f1)
  return em,F1

In [ ]:
results = {}

models = ["distilbert-base-uncased-distilled-squad",
    "bert-large-uncased-whole-word-masking-finetuned-squad",
    "deepset/roberta-base-squad2"
]
for model_name in models :
  em,f1 = evaluate_model(model_name,df)
  results[model_name] = {"Exact Match":em,"F1 Score":f1}

results_df = pd.DataFrame(results).T
print(results_df)

In [ ]:
def evaluate_model(model_name,df,n=50):
  print("Evaluate model",{model_name})
  qa_pipeline = pipeline("question-answering",model = model_name)

  exact_matches, F1_scores = [], []
  for i in range(n):
    row = df.iloc[i]
    result = qa_pipeline(context = row['context'],question = row['question'])
    pred = result['answer']
    true = row['answer']

    exact_matches.append(CAL_exact_match(pred,true))
    F1_scores.append(CAL_F1_score(pred,true))

  em = np.mean(exact_matches)
  f1 = np.mean(F1_scores)

  print("Exact Match :",em)
  print("F1 Score :",f1)
  return em, f1

In [ ]:
import streamlit as st
!pip install pyngrok

In [ ]:
st.set_page_config(
    page_title="Question Answering System",
    page_icon="❓",
    layout="wide"
)

st.title("🤖 Question Answering System")
st.markdown("""
This system uses transformer models to answer questions based on a given context.
Upload your context text and ask questions to get answers!
""")

st.sidebar.title("Configuration")
model_choice = st.sidebar.selectbox(
    "Choose QA Model:",
    [
        "distilbert-base-cased-distilled-squad",
        "bert-large-uncased-whole-word-masking-finetuned-squad",
        "deepset/roberta-base-squad2"
    ],
    help="Select which transformer model to use for question answering"
)

@st.cache_resource
def load_qa_model(model_name):
    """Load the QA model with error handling"""
    try:
        model = pipeline(
            "question-answering",
            model=model_name,
            device=0 if torch.cuda.is_available() else -1
        )
        return model
    except Exception as e:
        st.error(f"Error loading model: {e}")
        return pipeline("question-answering")

with st.spinner(f"Loading {model_choice}..."):
    qa_pipeline = load_qa_model(model_choice)

st.sidebar.success(f"Model loaded: {model_choice}")

col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("📝 Input Context")
    context = st.text_area(
        "Enter your context/passage:",
        height=200,
        placeholder="Paste the text passage that contains the information needed to answer your questions..."
    )

with col2:
    st.subheader("❓ Ask Questions")
    question = st.text_input(
        "Enter your question:",
        placeholder="What would you like to know about the context?"
    )

    st.markdown("**Example questions:**")
    st.markdown("- What is the main topic?")
    st.markdown("- Who is mentioned in the text?")
    st.markdown("- When did the event occur?")

if st.button("Get Answer", type="primary"):
    if not context.strip():
        st.warning("Please enter a context passage.")
    elif not question.strip():
        st.warning("Please enter a question.")
    else:
        with st.spinner("Analyzing text and finding answer..."):
            try:
                result = qa_pipeline(
                    context=context,
                    question=question
                )

                st.subheader("✅ Answer")
                st.success(result["answer"])

                st.metric("Confidence Score", f"{result['score']:.2%}")

                st.subheader("🔍 Answer in Context")
                start_idx = result['start']
                end_idx = result['end']

                before_answer = context[:start_idx]
                answer_text = context[start_idx:end_idx]
                after_answer = context[end_idx:]

                highlighted_text = f"{before_answer}<mark>{answer_text}</mark>{after_answer}"
                st.markdown(highlighted_text, unsafe_allow_html=True)

            except Exception as e:
                st.error(f"Error generating answer: {e}")

st.markdown("---")
st.subheader("📊 Batch Processing")

uploaded_file = st.file_uploader(
    "Upload a CSV file with contexts and questions",
    type=['csv'],
    help="CSV should have 'context' and 'question' columns"
)

if uploaded_file is not None:
    try:
        batch_df = pd.read_csv(uploaded_file)
        st.write("Preview of uploaded data:")
        st.dataframe(batch_df.head())

        if st.button("Process Batch"):
            progress_bar = st.progress(0)
            results = []

            for i, row in batch_df.iterrows():
                try:
                    result = qa_pipeline(
                        context=row['context'],
                        question=row['question']
                    )
                    results.append({
                        'context': row['context'],
                        'question': row['question'],
                        'answer': result['answer'],
                        'confidence': result['score']
                    })
                except Exception as e:
                    results.append({
                        'context': row['context'],
                        'question': row['question'],
                        'answer': f"Error: {e}",
                        'confidence': 0.0
                    })

                progress_bar.progress((i + 1) / len(batch_df))

            results_df = pd.DataFrame(results)
            st.subheader("Batch Results")
            st.dataframe(results_df)

            csv = results_df.to_csv(index=False)
            st.download_button(
                label="Download Results as CSV",
                data=csv,
                file_name="qa_results.csv",
                mime="text/csv"
            )

    except Exception as e:
        st.error(f"Error processing file: {e}")

st.markdown("---")
st.markdown(
    "Built with 🤗 Transformers | "
    "Models: DistilBERT, BERT, RoBERTa | "
    "Dataset: SQuAD v1.1"
)

In [ ]:
!streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]

In [ ]:
!streamlit run app.py